In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
from torch_geometric.utils import dense_to_sparse
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter


In [2]:
# --- 1. 超参数配置 ---
hparams = {
    'dataset': 'car',
    'threshold_pos': 500,
    'threshold_neg': 20000,
    'hidden_channels': 16,
    'heads': 4,
    'learning_rate': 0.005,
    'weight_decay': 5e-4,
    'epochs': 150,
    'dropout': 0.5,
    'seed': 42
}

torch.manual_seed(hparams['seed'])
np.random.seed(hparams['seed'])


In [3]:
# --- 2. TensorBoard 设置 ---
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
log_dir_name = f"../runs/{hparams['dataset']}_transformer_without_cpe_noleak_{timestamp}"
writer = SummaryWriter(log_dir_name)
print(f"TensorBoard 日志将保存在: {log_dir_name}")


TensorBoard 日志将保存在: ../runs/car_transformer_without_cpe_noleak_20260625-165040


In [4]:
# --- 3. 标签与边权处理函数 ---
def load_labels(base_path, dataset_name, expected_num_nodes):
    candidate_paths = [
        f"{base_path}{dataset_name}.data",
        f"{base_path}{dataset_name}.data.csv",
    ]

    for labels_path in candidate_paths:
        try:
            df = pd.read_csv(labels_path, header=None)
        except FileNotFoundError:
            continue

        df = df.dropna(axis=1, how='all')
        if len(df) != expected_num_nodes:
            continue
        return df.iloc[:, -1].values

    raise ValueError(f"无法读取与对象数量 {expected_num_nodes} 匹配的标签文件: {candidate_paths}")


def normalize_edge_attr(edge_attr):
    edge_attr = torch.log1p(edge_attr.float())
    min_value = edge_attr.min()
    max_value = edge_attr.max()
    if max_value > min_value:
        edge_attr = (edge_attr - min_value) / (max_value - min_value)
    return edge_attr


In [5]:
# --- 4. 数据加载与预处理函数 (不加入 CPE) ---
def load_and_prepare_data(dataset_name, threshold_pos, threshold_neg):
    base_path = f'../data/{dataset_name}/'

    features_path = f"{base_path}{dataset_name}.data.cleaned.csv"
    x_numpy = np.loadtxt(features_path, delimiter=',')
    x_features = torch.tensor(x_numpy, dtype=torch.float)
    num_nodes = x_features.shape[0]

    adj_matrix_pos_path = f"{base_path}{dataset_name}_A_plus_UG.csv"
    a_plus_pos_numpy = np.loadtxt(adj_matrix_pos_path, delimiter=',')
    if a_plus_pos_numpy.shape != (num_nodes, num_nodes):
        raise ValueError(f"正概念邻接矩阵尺寸应为 {(num_nodes, num_nodes)}，实际为 {a_plus_pos_numpy.shape}")
    a_plus_pos = torch.tensor(a_plus_pos_numpy, dtype=torch.float)
    a_plus_pos[a_plus_pos <= threshold_pos] = 0
    a_plus_pos.fill_diagonal_(0)
    edge_index_pos, edge_attr_pos = dense_to_sparse(a_plus_pos)
    edge_attr_pos = normalize_edge_attr(edge_attr_pos)

    adj_matrix_neg_path = f"{base_path}{dataset_name}_A_negative_UG.csv"
    a_plus_neg_numpy = np.loadtxt(adj_matrix_neg_path, delimiter=',')
    if a_plus_neg_numpy.shape != (num_nodes, num_nodes):
        raise ValueError(f"负概念邻接矩阵尺寸应为 {(num_nodes, num_nodes)}，实际为 {a_plus_neg_numpy.shape}")
    a_plus_neg = torch.tensor(a_plus_neg_numpy, dtype=torch.float)
    a_plus_neg[a_plus_neg <= threshold_neg] = 0
    a_plus_neg.fill_diagonal_(0)
    edge_index_neg, edge_attr_neg = dense_to_sparse(a_plus_neg)
    edge_attr_neg = normalize_edge_attr(edge_attr_neg)

    x_pos = x_features
    x_neg = x_features
    print(f"原始特征维度: {x_features.shape[1]}")
    print(f"正分支特征维度: {x_pos.shape[1]}")
    print(f"负分支特征维度: {x_neg.shape[1]}")

    labels_numpy = load_labels(base_path, dataset_name, num_nodes)

    encoder = LabelEncoder()
    y_numpy = encoder.fit_transform(labels_numpy)
    y = torch.tensor(y_numpy, dtype=torch.long)
    if num_nodes != len(y):
        raise ValueError(f"标签数量必须和对象数量一致: num_nodes={num_nodes}, labels={len(y)}")

    data = Data(x_pos=x_pos, x_neg=x_neg, y=y,
                edge_index_pos=edge_index_pos, edge_attr_pos=edge_attr_pos.view(-1, 1),
                edge_index_neg=edge_index_neg, edge_attr_neg=edge_attr_neg.view(-1, 1),
                num_nodes=num_nodes)

    num_train = int(num_nodes * 0.6)
    num_val = int(num_nodes * 0.2)
    indices = torch.randperm(num_nodes)
    data.train_mask = torch.zeros(num_nodes, dtype=torch.bool); data.train_mask[indices[:num_train]] = True
    data.val_mask = torch.zeros(num_nodes, dtype=torch.bool); data.val_mask[indices[num_train:num_train + num_val]] = True
    data.test_mask = torch.zeros(num_nodes, dtype=torch.bool); data.test_mask[indices[num_train + num_val:]] = True

    return data, len(np.unique(y_numpy))


In [6]:
# --- 5. 定义模型 ---
class DualConceptTransformer(nn.Module):
    def __init__(self, pos_in_channels, neg_in_channels, hidden_channels, out_channels, heads=1, dropout=0.5):
        super(DualConceptTransformer, self).__init__()
        self.dropout = dropout
        self.pos_conv = TransformerConv(pos_in_channels, hidden_channels, heads=heads, edge_dim=1)
        self.neg_conv = TransformerConv(neg_in_channels, hidden_channels, heads=heads, edge_dim=1)
        self.fusion_layer = nn.Linear(hidden_channels * heads * 2, out_channels)

    def forward(self, x_pos, x_neg, edge_index_pos, edge_attr_pos, edge_index_neg, edge_attr_neg):
        h_pos = self.pos_conv(x_pos, edge_index_pos, edge_attr_pos)
        h_pos = F.relu(h_pos)
        h_pos = F.dropout(h_pos, p=self.dropout, training=self.training)

        h_neg = self.neg_conv(x_neg, edge_index_neg, edge_attr_neg)
        h_neg = F.relu(h_neg)
        h_neg = F.dropout(h_neg, p=self.dropout, training=self.training)

        h_combined = torch.cat([h_pos, h_neg], dim=1)
        out = self.fusion_layer(h_combined)
        return out


In [7]:
# --- 6. 实例化数据和模型 ---
data, num_classes = load_and_prepare_data(hparams['dataset'],
                                          hparams['threshold_pos'],
                                          hparams['threshold_neg'])

model = DualConceptTransformer(pos_in_channels=data.x_pos.shape[1],
                               neg_in_channels=data.x_neg.shape[1],
                               hidden_channels=hparams['hidden_channels'],
                               out_channels=num_classes,
                               heads=hparams['heads'],
                               dropout=hparams['dropout'])

optimizer = torch.optim.Adam(model.parameters(), lr=hparams['learning_rate'], weight_decay=hparams['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()


原始特征维度: 21
正分支特征维度: 21
负分支特征维度: 21


In [8]:
# --- 7. 训练与评估函数 ---
def train(epoch):
    model.train()
    optimizer.zero_grad()
    out = model(data.x_pos, data.x_neg, data.edge_index_pos, data.edge_attr_pos, data.edge_index_neg, data.edge_attr_neg)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    writer.add_scalar('Loss/train', loss.item(), epoch)
    return loss.item()

def evaluate(epoch):
    model.eval()
    with torch.no_grad():
        out = model(data.x_pos, data.x_neg, data.edge_index_pos, data.edge_attr_pos, data.edge_index_neg, data.edge_attr_neg)
        pred = out.argmax(dim=1)

        train_acc = (pred[data.train_mask] == data.y[data.train_mask]).sum().item() / data.train_mask.sum().item()
        val_acc = (pred[data.val_mask] == data.y[data.val_mask]).sum().item() / data.val_mask.sum().item()
        test_acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()

        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/validation', val_acc, epoch)
        writer.add_scalar('Accuracy/test', test_acc, epoch)

        return train_acc, val_acc, test_acc


In [9]:
# --- 8. 主训练循环 ---
print("\n--- 开始训练 (不带 CPE 的双概念格 Graph Transformer) ---")
for epoch in range(1, hparams['epochs'] + 1):
    loss = train(epoch)
    if epoch % 1 == 0:
        train_acc, val_acc, test_acc = evaluate(epoch)
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}, Test Acc: {test_acc:.4f}')

final_train_acc, final_val_acc, final_test_acc = evaluate(hparams['epochs'])
print('--- 训练完成 ---')
print(f'最终测试集准确率: {final_test_acc:.4f}')

metrics = {'accuracy/final_train': final_train_acc, 'accuracy/final_validation': final_val_acc, 'accuracy/final_test': final_test_acc}
writer.add_hparams(hparams, metrics)
writer.close()



--- 开始训练 (不带 CPE 的双概念格 Graph Transformer) ---


Epoch: 001, Loss: 1.2449, Train Acc: 0.7075, Val Acc: 0.6986, Test Acc: 0.7205


Epoch: 002, Loss: 1.0880, Train Acc: 0.7037, Val Acc: 0.6870, Test Acc: 0.7205
Epoch: 003, Loss: 0.9538, Train Acc: 0.7017, Val Acc: 0.6870, Test Acc: 0.7147


Epoch: 004, Loss: 0.8485, Train Acc: 0.7008, Val Acc: 0.6870, Test Acc: 0.7147


Epoch: 005, Loss: 0.7718, Train Acc: 0.7017, Val Acc: 0.6870, Test Acc: 0.7147
Epoch: 006, Loss: 0.7134, Train Acc: 0.7037, Val Acc: 0.6870, Test Acc: 0.7205


...

Epoch: 138, Loss: 0.0288, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 139, Loss: 0.0302, Train Acc: 0.9990, Val Acc: 0.9797, Test Acc: 1.0000
Epoch: 140, Loss: 0.0265, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000


Epoch: 141, Loss: 0.0277, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 142, Loss: 0.0265, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 143, Loss: 0.0274, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000


Epoch: 144, Loss: 0.0233, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 145, Loss: 0.0234, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 146, Loss: 0.0298, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000


Epoch: 147, Loss: 0.0260, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 148, Loss: 0.0268, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
Epoch: 149, Loss: 0.0265, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000


Epoch: 150, Loss: 0.0243, Train Acc: 0.9990, Val Acc: 0.9826, Test Acc: 1.0000
--- 训练完成 ---
最终测试集准确率: 1.0000
